# EcoSort: Intelligent Waste Management Assistant — Summative Lab (NNs & Similar Models)

**Part 1:** Dataset exploration and preparation  
**Part 2:** Waste material classification with a CNN (transfer learning)  
**Part 3:** Waste description classification (text)  
**Part 4:** Recycling instruction generation with RAG  
**Part 5:** Integrated waste management assistant  

Authoring environment: Windows | Python 3.12 | CPU only (16 GB RAM)


In [ ]:
# %% [markdown]
# ## 0. Global configuration
# We centralise every path, class name, and hyper-parameter here so the whole
# notebook stays consistent and reproducible.

import os
import sys
import json
import re
import math
import random
import time
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

# Quiet noisy TF startup logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import tensorflow as tf
try:
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        for g in gpus:
            tf.config.experimental.set_memory_growth(g, True)
    else:
        print('INFO: No GPU detected - running on CPU.')
except Exception as e:
    print('GPU config skipped:', e)
tf.random.set_seed(42)

import torch
torch.manual_seed(42)

# ---------------------------------------------------------------- paths
BASE = os.getcwd()
DATA = os.path.join(BASE, 'data')          # derived data (splits, features)
FIG = os.path.join(DATA, 'figures')        # saved plots
MODELS = os.path.join(BASE, 'models')      # saved model artifacts
os.makedirs(DATA, exist_ok=True)
os.makedirs(FIG, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)

def find_realwaste_root():
    cands = [
        os.path.join(BASE, 'realwaste', 'realwaste-main', 'RealWaste'),
        os.path.join(BASE, 'RealWaste'),
        os.path.join(BASE, 'data', 'RealWaste'),
    ]
    for c in cands:
        if os.path.isdir(c):
            return c
    raise FileNotFoundError('RealWaste image root not found. Please place the '
                            '"RealWaste" class folders next to the data.')
IMAGE_ROOT = find_realwaste_root()
TEXT_CSV = os.path.join(BASE, 'waste_descriptions.csv')
POLICY_JSON = os.path.join(BASE, 'waste_policy_documents.json')

CLASSES = sorted([d for d in os.listdir(IMAGE_ROOT)
                  if os.path.isdir(os.path.join(IMAGE_ROOT, d))])
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
print(f'IMAGE_ROOT  : {IMAGE_ROOT}')
print(f'N classes   : {len(CLASSES)}')
print(f'Classes     : {CLASSES}')
print(f'TensorFlow  : {tf.__version__}   torch: {torch.__version__}')


In [ ]:
# %% [markdown]
# ### Shared helpers (used everywhere below)

def seed_everything(s=42):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s); torch.manual_seed(s)

def save_fig(fig, name, show=True, dpi=110, tight=True):
    """Save the figure and (for the notebook output) display it."""
    p = os.path.join(FIG, name)
    fig.savefig(p, dpi=dpi, bbox_inches='tight' if tight else None)
    if show:
        plt.show()
    plt.close(fig)
    return p

def clf_metrics(y_true, y_pred, labels):
    from sklearn.metrics import (accuracy_score, precision_score,
                                 recall_score, f1_score, classification_report,
                                 confusion_matrix)
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'report': classification_report(y_true, y_pred, target_names=labels, zero_division=0, digits=3),
        'confusion': confusion_matrix(y_true, y_pred, labels=range(len(labels))),
    }

def plot_confusion(cm, labels, title='Confusion matrix', cmap='Blues'):
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, xticklabels=labels,
                yticklabels=labels, ax=ax, cbar=False)
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    return fig


# Part 1: Dataset Exploration & Preparation

## 1.1 RealWaste image dataset

We start by building a tidy DataFrame over the **RealWaste** images
(9 material classes), scanning basic metadata (size, colour mode, file
size, brightness / saturation) so we can reason about the visual
characteristics and the challenges the CNN will face.


In [ ]:
# -------------------------------------------------
# 1.1.1 Build the image catalogue (cached on disk)
meta_path = os.path.join(DATA, 'image_metadata.csv')
if os.path.exists(meta_path):
    df_img = pd.read_csv(meta_path)
    print('Loaded cached image metadata.')
else:
    rows = []
    for cls in CLASSES:
        cdir = os.path.join(IMAGE_ROOT, cls)
        for fn in sorted(os.listdir(cdir)):
            p = os.path.join(cdir, fn)
            rows.append({'filename': fn, 'class': cls, 'path': p})
    df_img = pd.DataFrame(rows)

    def scan_row(r):
        try:
            im = Image.open(r['path'])
            w, h = im.size
            mode = im.mode
            fs = os.path.getsize(r['path'])
            small = im.convert('RGB').resize((48, 48))
            arr = np.asarray(small).astype(np.float32)
            hsv = np.asarray(small.convert('HSV')).astype(np.float32)
            im.close()
            return pd.Series({
                'width': w, 'height': h, 'mode': mode,
                'aspect_ratio': w / h, 'file_kb': round(fs / 1024, 1),
                'mean_brightness': arr.mean(),
                'std_brightness': arr.std(),
                'mean_saturation': hsv[..., 1].mean(),
            })
        except Exception as ex:
            return pd.Series({'width': np.nan, 'height': np.nan, 'mode': 'ERR',
                              'aspect_ratio': np.nan, 'file_kb': np.nan,
                              'mean_brightness': np.nan, 'std_brightness': np.nan,
                              'mean_saturation': np.nan})

    meta = df_img.apply(scan_row, axis=1)
    df_img = pd.concat([df_img, meta], axis=1)
    df_img.to_csv(meta_path, index=False)
    print('Catalogue written to', meta_path)

print(df_img.shape)
print('Classes:', df_img.groupby('class').size().to_dict())
print('NaNs in metadata:', df_img[['width', 'height', 'mean_brightness']].isna().sum().sum())


In [ ]:
# -------------------------------------------------
# 1.1.2 Class distribution
class_counts = df_img['class'].value_counts().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.bar(class_counts.index, class_counts.values, color=sns.color_palette('viridis', len(CLASSES)))
ax.set_title('RealWaste - number of images per material class')
ax.set_ylabel('Images')
for b, v in zip(bars, class_counts.values):
    ax.text(b.get_x() + b.get_width() / 2, v + 4, str(v), ha='center', fontsize=9)
plt.xticks(rotation=45, ha='right')
save_fig(fig, 'image_class_distribution.png')

imbalance = class_counts.max() / class_counts.min()
print(f'Most frequent class: {class_counts.index[0]} ({class_counts.iloc[0]})')
print(f'Least frequent class: {class_counts.index[-1]} ({class_counts.iloc[-1]})')
print(f'Class imbalance ratio (max/min) = {imbalance:.2f}x  -> class weights will be used.')

# -------------------------------------------------
# 1.1.3 Image geometry / quality summary
print('\n--- Image geometry  (all sizes are square) ---')
print(df_img[['width', 'height', 'aspect_ratio', 'file_kb']].describe().round(1).to_string())
print('\nColour modes:', df_img['mode'].value_counts().to_dict())

print('\n--- Per-class visual summaries (brightness 0-255, saturation 0-255) ---')
g = df_img.groupby('class')[['mean_brightness', 'mean_saturation', 'std_brightness', 'file_kb']].mean().round(1).sort_values('mean_brightness')
print(g.to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df_img, x='class', y='mean_brightness', ax=axes[0], palette='viridis')
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_title('Mean brightness per class')
sns.boxplot(data=df_img, x='class', y='mean_saturation', ax=axes[1], palette='magma')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_title('Mean saturation per class')
save_fig(fig, 'image_brightness_saturation.png')


In [ ]:
# -------------------------------------------------
# 1.1.4 Look at real samples per class
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for ax, cls in zip(axes.ravel(), CLASSES):
    sub = df_img[df_img['class'] == cls]
    sample = sub.iloc[random.randrange(len(sub))]['path']
    im = Image.open(sample).resize((180, 180))
    ax.imshow(im); ax.set_title(cls, fontsize=10); ax.axis('off')
plt.suptitle('Random sample from each RealWaste class', y=1.01)
save_fig(fig, 'image_class_samples.png')

# And one richer grid for the "Plastic vs Glass vs Metal" family which we
# expect to be confusable.
def grid_for(cls_subset, title, n=6):
    rng = random.Random(7)
    rows = []
    for cls in cls_subset:
        sub = df_img[df_img['class'] == cls]
        rows.append(sub.sample(n, random_state=7))
    panel = pd.concat(rows)
    fig, axes = plt.subplots(n, len(cls_subset), figsize=(3 * len(cls_subset), 3 * n))
    for i, cls in enumerate(cls_subset):
        for j, (_, r) in enumerate(panel[panel['class'] == cls].iterrows()):
            im = Image.open(r['path']).resize((120, 120))
            axes[j, i].imshow(im); axes[j, i].axis('off')
            if j == 0:
                axes[j, i].set_title(cls, fontsize=10)
    plt.suptitle(title, y=1.01)
    return fig

fig = grid_for(['Plastic', 'Glass', 'Metal', 'Miscellaneous Trash'],
               'Potentially confusable classes: Plastic vs Glass vs Metal vs Miscellaneous Trash')
save_fig(fig, 'image_confusable_grid.png')


## 1.2 Waste description text dataset (`waste_descriptions.csv`)

5,000 resident-style descriptions of waste items with:
`description`, `category`, `disposal_instruction`, `common_confusion`,
`material_composition`.


In [ ]:
# -------------------------------------------------
# 1.2.1 Load and inspect the description data
df_txt = pd.read_csv(TEXT_CSV)
print('Shape:', df_txt.shape)
print('\nDtypes:\n', df_txt.dtypes)
print('\nHead:')
print(df_txt.head(8).to_string())
print('\nMissing values:')
print(df_txt.isna().sum().to_string())

print('\nCategory distribution:')
ct = df_txt['category'].value_counts()
print(ct.to_string())

fig, ax = plt.subplots(figsize=(10, 4.5))
ct.sort_values().plot.barh(ax=ax, color=sns.color_palette('viridis', len(CLASSES)))
ax.set_title('waste_descriptions.csv - category distribution')
ax.set_xlabel('Count')
plt.tight_layout()
save_fig(fig, 'text_class_distribution.png')

# -------------------------------------------------
# 1.2.2 Vocabulary & structure analysis
lengths = df_txt['description'].str.split().str.len()
print('\nDescription word-count: mean %.2f, std %.2f, min %d, max %d' %
      (lengths.mean(), lengths.std(), lengths.min(), lengths.max()))

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(lengths, bins=range(0, 13), edgecolor='k', color='steelblue')
ax.set_title('Description length (words)')
ax.set_xlabel('words'); ax.set_ylabel('count')
save_fig(fig, 'text_description_length.png')

vocab = set()
df_txt['description'].str.lower().str.findall(r"[a-z']+").apply(lambda t: vocab.update(t))
print(f'Unique lowercase tokens across all descriptions: {len(vocab):,}')

# -------------------------------------------------
from sklearn.feature_extraction.text import CountVectorizer
STOP = set(__import__('sklearn.feature_extraction.text', fromlist=['ENGLISH_STOP_WORDS']).ENGLISH_STOP_WORDS)
top_words = {}
for cls in CLASSES:
    corpus = df_txt[df_txt['category'] == cls]['description'].tolist()
    cv = CountVectorizer(ngram_range=(1, 1), stop_words='english')
    try:
        X = cv.fit_transform(corpus)
    except ValueError:
        continue
    sums = np.asarray(X.sum(axis=0)).ravel()
    idx = np.argsort(sums)[::-1][:8]
    top_words[cls] = [(cv.get_feature_names_out()[i], int(sums[i])) for i in idx]

print('\nTop-8 characteristic tokens per class:')
for cls, toks in top_words.items():
    print(f'{cls:20s} -> ' + ', '.join(f'{w}({c})' for w, c in toks))


In [ ]:
# -------------------------------------------------
# 1.2.3 Auxiliary text fields (why the extra columns matter for RAG)
print('common_confusion - filled in', df_txt['common_confusion'].notna().sum(), 'of', len(df_txt), 'rows')
print('\nMost frequent confusion notes:')
cc = df_txt['common_confusion'].dropna()
print(Counter(cc).most_common(8))

print('\nUnique disposal instructions:', df_txt['disposal_instruction'].nunique())
print('Most frequent disposal instructions:')
dc = df_txt['disposal_instruction'].value_counts()
print(dc.head(8).to_string())

print('\nUnique material-composition strings:', df_txt['material_composition'].nunique())
mc = df_txt['material_composition'].value_counts()
print(mc.head(8).to_string())

print('\nSample rows for the ambiguous class "Miscellaneous Trash":')
print(df_txt[df_txt['category'] == 'Miscellaneous Trash'].sample(5, random_state=1)[
    ['description', 'disposal_instruction', 'common_confusion']].to_string())


## 1.3 Waste policy documents (`waste_policy_documents.json`)

Municipal regulations from *Metro City*. Each record has a `policy_type`,
`categories_covered`, `effective_date`, `jurisdiction` and the raw
`document_text` block. These documents become both the **retrieval corpus**
and the **supervision signal** for the RAG generator.


In [ ]:
# -------------------------------------------------
# 1.3.1 Load and organise the policy documents
with open(POLICY_JSON, 'r', encoding='utf-8') as f:
    policy_raw = json.load(f)
print('Number of policy documents:', len(policy_raw))
pf = pd.DataFrame(policy_raw)
print('\nColumns:', list(pf.columns))
print('\nHead of tabular fields:')
print(pf[['policy_id', 'policy_type', 'categories_covered', 'effective_date',
          'jurisdiction']].head(14).to_string())

pf['n_words'] = pf['document_text'].str.split().str.len()
pf['n_lines'] = pf['document_text'].str.count('\n') + 1
print('\nDocument length stats (words):')
print(pf['n_words'].describe().round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
pf['n_words'].plot.hist(bins=15, ax=axes[0], edgecolor='k', color='seagreen')
axes[0].set_title('Policy document length (words)')
pf['policy_type'].value_counts().sort_values()[:12].plot.barh(ax=axes[1], color='teal')
axes[1].set_title('Policy types')
plt.tight_layout()
save_fig(fig, 'policy_docs_summary.png')

cov = Counter()
for cl in pf['categories_covered']:
    for c in cl:
        cov[c] += 1
print('\nPolicies covering each category:')
print(pd.Series(cov).sort_values(ascending=False).to_string())

# -------------------------------------------------
# 1.3.2 Structure of a document (they use a recurring SKELETON)
print('\n--- Structure of a typical policy document ---')
print(policy_raw[3]['document_text'][:900])


### 1.4 Preliminary data-quality findings & challenges

**Images (RealWaste)**
1. **Uniform geometry** — every image is an RGB JPEG, exactly `524 x 524`.
   There is no resolution variation to handle, but real resident photos will
   differ; we therefore still use augmentation (flips, rotations, zoom) so the
   CNN does not overfit to the dataset's particular capture style.
2. **Class imbalance** — `Plastic` (921) is ~2.9x more frequent than
   `Textile Trash` (318). Vanilla training will bias toward frequent classes,
   so we balance the loss with *class weights*.
3. **Confusable material families** — transparent bottles mean *Plastic vs
   Glass* and *Paper vs Cardboard* look similar; *Miscellaneous Trash* is a
   catch-all with internally inconsistent visuals. Expect confusion along
   these pairs; we analyse this explicitly.
4. **Background clutter** — capture-style differences (napkins, hands,
   tables) act as domain noise; brightness/saturation vary strongly which is
   why colour augmentation matters.

**Text (`waste_descriptions.csv`)**
1. **Very short inputs** (mean ~5 words) — almost the whole signal lives in a
   handful of tokens (e.g. "bottle", "crushed", "grounds").
2. **Half of `common_confusion` is empty** (2,504 blanks) — we treat it as an
   optional pre-retrieval hint, never a required field.
3. **Adjective-led phrasing** ("soiled", "wrinkled", "intact") is irrelevant
   to material class and adds noise; we keep noun vocabulary but watch that
   TF-IDF does not latch onto these modifiers.
4. Regional/term variation (see `material_composition`) needs normalisation so
   the classifier and retriever agree.

**Policy documents (JSON)**
1. A small corpus (14 docs) but with a highly *regular skeleton*
   (ACCEPTABLE / NON-ACCEPTABLE / COLLECTION METHOD / PREPARATION /
   BENEFITS) which lets us chunk reliably.
2. Some documents cover **multiple categories** (e.g. Municipal Waste) and a
   few categories are *not* covered by their own dedicated document
   (e.g. Glass colours) — retrieval must therefore handle multi-category
   sections.
3. `categories_covered` gives us *ground-truth tags* to score retrieval.

These findings drive the preprocessing decisions made in the rest of Part 1.


### 1.5 Data pipelines for every modality

Now we create tidy, reproducible artifacts used by Parts 2-4:

* **Image split** — stratified 70 / 15 / 15 into `data/img_*.csv`, plus a
  `tf.data` pipeline (resize + augment + normalise).
* **Text split** — stratified 80 / 10 / 10 into `data/txt_*.csv` together with
  a light cleaning function used by both classical and neural models.
* **Policy chunking** — the 14 documents are split into per-category sections
  stored in `data/policy_chunks.csv`, ready to become RAG retrieval units.


In [ ]:
# ------------------------------------------------------
# 1.5.1 Image split (stratified) + tf.data pipeline
from sklearn.model_selection import train_test_split

train_df, tmp = train_test_split(df_img, test_size=0.30, stratify=df_img['class'],
                                 random_state=42)
val_df, test_df = train_test_split(tmp, test_size=0.50, stratify=tmp['class'],
                                   random_state=42)
for name, d in [('img_train', train_df), ('img_val', val_df), ('img_test', test_df)]:
    d[['path', 'class', 'filename']].to_csv(os.path.join(DATA, name + '.csv'), index=False)
print('Image split sizes:', len(train_df), len(val_df), len(test_df))
print('Stratification check (class balances in each split):')
print(pd.DataFrame({
    'train': train_df['class'].value_counts(normalize=True),
    'val':   val_df['class'].value_counts(normalize=True),
    'test':  test_df['class'].value_counts(normalize=True),
}).round(3).to_string())

y_train_img = train_df['class'].map(CLASS_TO_IDX).values
class_weight_dict = {}
for cur in np.unique(y_train_img):
    n = (y_train_img == cur).sum()
    class_weight_dict[cur] = len(y_train_img) / (len(np.unique(y_train_img)) * n)
print('\nComputed image class weights (inverse frequency, normalised):')
print({IDX_TO_CLASS[k]: round(v, 3) for k, v in class_weight_dict.items()})

IMG_SIZE_DEFAULT = 224

def build_img_pipeline(df, img_size, batch_size, augment=False, cache_name=None):
    paths = df['path'].values
    labels = df['class'].map(CLASS_TO_IDX).values.astype(np.int64)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    def _load(p, y):
        raw = tf.io.read_file(p)
        x = tf.image.decode_jpeg(raw, channels=3)
        x = tf.image.resize(x, (img_size, img_size))
        return x, y
    if augment:
        aug = tf.keras.Sequential([
            tf.keras.layers.RandomFlip('horizontal'),
            tf.keras.layers.RandomRotation(0.15),
            tf.keras.layers.RandomZoom(0.15),
            tf.keras.layers.RandomBrightness(0.15),
            tf.keras.layers.RandomContrast(0.15),
        ], name='aug')
        def _a(x, y):
            x = tf.cast(x, tf.float32)
            return aug(x, training=True), y
    else:
        def _a(x, y):
            return tf.cast(x, tf.float32), y
    def _norm(x, y):
        x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
        return tf.cast(x, tf.float32), y
    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(_a, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(_norm, num_parallel_calls=tf.data.AUTOTUNE)
    if cache_name:
        ds = ds.cache(os.path.join(DATA, cache_name))
    ds = ds.shuffle(2048, reshuffle_each_iteration=True) if augment else ds
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds224 = build_img_pipeline(train_df, IMG_SIZE_DEFAULT, 32, augment=True, cache_name='cache_train_224')
val_ds224   = build_img_pipeline(val_df,   IMG_SIZE_DEFAULT, 32, augment=False, cache_name='cache_val_224')
test_ds224  = build_img_pipeline(test_df,  IMG_SIZE_DEFAULT, 32, augment=False, cache_name='cache_test_224')
print('Sample batch shapes (train):',
      [t.shape for t in next(iter(train_ds224.unbatch().batch(8)))])

# ------------------------------------------------------
# 1.5.2 Text cleaning + split
def clean_text(s):
    """Lightweight cleaning tuned for resident-style descriptions."""
    if not isinstance(s, str):
        return ''
    s = s.lower()
    s = re.sub(r'https?://\S+|www\.\S+', ' ', s)      # urls
    s = re.sub(r'[\w.+-]+@[\w-]+\.[\w.-]+', ' ', s)   # emails
    s = re.sub(r'[^a-z\s]', ' ', s)                   # punctuation/numbers
    s = re.sub(r'\s+', ' ', s).strip()
    return s

df_txt['clean'] = df_txt['description'].apply(clean_text)
df_txt['n_words'] = df_txt['clean'].str.split().str.len()
print('Clean examples:')
print(df_txt[['description', 'clean']].head(5).to_string())

X_txt = df_txt['clean'].values.astype(str)
y_txt = df_txt['category'].astype('category').cat.codes.values
cat_names = list(df_txt['category'].astype('category').cat.categories)
for i, c in enumerate(cat_names):
    print(i, c)

Xtr, tmpX = train_test_split(np.arange(len(df_txt)), test_size=0.20, stratify=y_txt, random_state=42)
Xv, Xte = train_test_split(tmpX, test_size=0.50, stratify=y_txt[tmpX], random_state=42)
train_idx, val_idx, test_idx = Xtr, Xv, Xte
print('\nText split sizes:', len(train_idx), len(val_idx), len(test_idx))
for name, idx in [('txt_train', train_idx), ('txt_val', val_idx), ('txt_test', test_idx)]:
    pd.DataFrame({'index': idx, 'text': df_txt['clean'].iloc[idx].values,
                  'label': df_txt['category'].iloc[idx].values}).to_csv(
        os.path.join(DATA, name + '.csv'), index=False)

# ------------------------------------------------------
# 1.5.3 Policy documents -> per-category chunks (improved splitter)
def split_sections(doc_text):
    """Split a policy document on its recurring section headers."""
    lines = doc_text.split('\n')
    header_re = re.compile(r'^[A-Za-z][A-Za-z0-9 /\-]*:$')
    sections = []
    cur_head, cur = None, []
    for ln in lines:
        s = ln.strip()
        if s and len(s) <= 45 and header_re.match(s) and not s.startswith('-'):
            if cur_head is not None:
                sections.append((cur_head, '\n'.join(cur).strip()))
            cur_head, cur = s.rstrip(':').strip(), []
        else:
            cur.append(ln)
    if cur_head is not None:
        sections.append((cur_head, '\n'.join(cur).strip()))
    return sections

def assign_category(header, categories):
    h = header.lower()
    if len(categories) == 1:
        return categories[0]
    for c in categories:
        if c.lower() in h:
            return c
    return None  # e.g. GENERAL REQUIREMENTS section

CHUNKS = []
for doc in policy_raw:
    for header, body in split_sections(doc['document_text']):
        tag = assign_category(header, doc['categories_covered'])
        CHUNKS.append({
            'section': header, 'category': tag, 'text': body,
            'policy_id': doc['policy_id'], 'policy_type': doc['policy_type'],
            'categories_covered': ','.join(doc['categories_covered']),
            'effective_date': doc['effective_date'], 'jurisdiction': doc['jurisdiction'],
        })
chunk_df = pd.DataFrame(CHUNKS)
print('Policy chunks created:', chunk_df.shape)
print('\nTagged vs untagged chunks:')
print(chunk_df['category'].fillna('GENERAL').value_counts().to_string())
print('\nA few sample chunks:')
print(chunk_df[['section', 'category', 'text']].head(3).to_string())
print('\nChunk word-count stats:')
print(chunk_df['text'].str.split().str.len().describe().round(1).to_string())
chunk_df.to_csv(os.path.join(DATA, 'policy_chunks.csv'), index=False)


# Part 3: Waste Description Classification (text)

Descriptions are **very short** (mean ~5 words) and use a limited
vocabulary, so a dense TF-IDF representation is extremely effective.
We follow a two-track design:

1. **Classical ML track** (Option A in the brief) — Multinomial Naive Bayes,
   Logistic Regression and Random Forest over TF-IDF features, with a small
   hyper-parameter search (ngram range, regularisation) as evidence of
   experimentation.
2. **Neural track** — a compact learned-embedding *BiLSTM* classifier handled
   by Keras TextVectorization; it learns task-specific word embeddings
   directly.

The best system (by validation accuracy) is wrapped in a reusable
`classify_description(text)` function. A note: on a GPU, fine-tuning
**DistilBERT** on the same 5,000 examples is the natural transformer upgrade
to this pipeline, but on our CPU budget the BiLSTM + strong TF-IDF baselines
are the right engineering trade-off.


In [ ]:
# --------------------------------------------------------
# 3.1 Build TF-IDF features (fit ONLY on the training folds)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score

X_train_txt = df_txt['clean'].iloc[train_idx].values
y_train_txt = y_txt[train_idx]
X_val_txt   = df_txt['clean'].iloc[val_idx].values
y_val_txt   = y_txt[val_idx]
X_test_txt  = df_txt['clean'].iloc[test_idx].values
y_test_txt  = y_txt[test_idx]

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95,
                        sublinear_tf=True, stop_words='english')
Xtr_t = tfidf.fit_transform(X_train_txt)
Xva_t = tfidf.transform(X_val_txt)
Xte_t = tfidf.transform(X_test_txt)
print('TF-IDF matrix (train):', Xtr_t.shape)
print('Vocab size:', len(tfidf.vocabulary_))

models = {
    'MultinomialNB': MultinomialNB(alpha=0.1),
    'LogisticRegression': LogisticRegression(C=1.0, max_iter=3000, n_jobs=-1),
    'RandomForest': RandomForestClassifier(n_estimators=250, n_jobs=-1, random_state=42),
}
val_accs = {}
for name, mdl in models.items():
    mdl.fit(Xtr_t, y_train_txt)
    va = accuracy_score(y_val_txt, mdl.predict(Xva_t))
    val_accs[name] = round(va, 4)
    print(f'{name:20s} val_acc = {va:.4f}')


In [ ]:
# --------------------------------------------------------
# 3.2 Hyper-parameter optimisation (evidence of experimentation)
param_grid = {
    'C': [0.1, 1.0, 10.0],
}
grid = GridSearchCV(LogisticRegression(max_iter=3000, n_jobs=-1), param_grid,
                    cv=StratifiedKFold(3, shuffle=True, random_state=42),
                    scoring='accuracy', n_jobs=-1, verbose=1)
grid.fit(Xtr_t, y_train_txt)
print('\nBest C for LogisticRegression:', grid.best_params_, 'cv_acc=%.4f' % grid.best_score_)

# Does character information add anything on top of word TF-IDF?
tfidf_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=3, sublinear_tf=True)
Xtr_c = tfidf_char.fit_transform(X_train_txt)
Xva_c = tfidf_char.transform(X_val_txt)

from scipy.sparse import hstack
Xtr_h = hstack([Xtr_t, Xtr_c])
Xva_h = hstack([Xva_t, Xva_c])
for name, Xtr, Xva in [('word-level only', Xtr_t, Xva_t), ('word + char(3-5)', Xtr_h, Xva_h)]:
    lr = LogisticRegression(C=grid.best_params_['C'], max_iter=3000, n_jobs=-1)
    lr.fit(Xtr, y_train_txt)
    print(f'LogisticRegression on {name:20s} -> val_acc = {accuracy_score(y_val_txt, lr.predict(Xva)):.4f}')

# Winner so far: fine-tuned Logistic Regression on word TF-IDF
final_ml = LogisticRegression(C=grid.best_params_['C'], max_iter=3000, n_jobs=-1)
final_ml.fit(Xtr_t, y_train_txt)
print('\nChosen classical model trained on word TF-IDF.')


In [ ]:
# --------------------------------------------------------
# 3.3 Neural text classifier (embeddings + BiLSTM)
from tensorflow.keras.layers import TextVectorization, Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping as ES2

from sklearn.utils.class_weight import compute_class_weight

SEQLEN = 22
vectorizer = TextVectorization(max_tokens=9000, output_mode='int',
                               output_sequence_length=SEQLEN)
vectorizer.adapt(X_train_txt.tolist())
print('Neural vocab size used:', vectorizer.vocabulary_size())

def build_bi_lstm(vocab_size, seqlen):
    m = Sequential([
        Embedding(vocab_size, 128, mask_zero=True, input_length=seqlen),
        Bidirectional(LSTM(64, dropout=0.3, return_sequences=False)),
        Dropout(0.3),
        Dense(9, activation='softmax'),
    ])
    return m

lstm_model = build_bi_lstm(vectorizer.vocabulary_size(), SEQLEN)
lstm_model.compile(Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

Xtr_n = vectorizer(X_train_txt.tolist())
Xva_n = vectorizer(X_val_txt.tolist())
Xte_n = vectorizer(X_test_txt.tolist())

cw_text = compute_class_weight('balanced', classes=np.unique(y_train_txt), y=y_train_txt)
cw_text_dict = {int(k): float(v) for k, v in zip(np.unique(y_train_txt), cw_text)}

lstm_model.fit(Xtr_n, y_train_txt, validation_data=(Xva_n, y_val_txt),
               epochs=14, batch_size=64, class_weight=cw_text_dict,
               callbacks=[ES2(monitor='val_accuracy', patience=3, restore_best_weights=True)],
               verbose=1)
val_nn = lstm_model.evaluate(Xva_n, y_val_txt, verbose=0)[1]
print(f'\nBiLSTM  val_acc = {val_nn:.4f}')

# save both winner artifacts
import joblib
joblib.dump({"model": final_ml, "tfidf": tfidf, "classes": list(CLASSES),
             "y_cat_names": list(df_txt['category'].astype('category').cat.categories)},
            os.path.join(MODELS, 'text_ml_pipeline.pkl'))
lstm_model.save(os.path.join(MODELS, 'text_bilstm.keras'))
print('Saved text model artifacts.')


In [ ]:
# --------------------------------------------------------
# 3.4 Evaluate the best text model on the held-out test set
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
best_txt = final_ml
yhat = best_txt.predict(Xte_t)
acc = accuracy_score(y_test_txt, yhat)
print(f'Final text classifier (LogisticRegression + TF-IDF): test accuracy = {acc:.4f}')

yhat_nn = np.argmax(lstm_model.predict(Xte_n, verbose=0), axis=1)
nn_te = accuracy_score(y_test_txt, yhat_nn)
print(f'Text classifier (BiLSTM):                test accuracy = {nn_te:.4f}')

cat_names = list(df_txt['category'].astype('category').cat.categories)
cm = confusion_matrix(y_test_txt, yhat, labels=range(len(cat_names)))
fig = plot_confusion(cm, cat_names, title='Text classification confusion matrix (LogReg + TF-IDF)')
save_fig(fig, 'text_confusion.png')

print('\nClassification report (best text model):')
print(classification_report(y_test_txt, yhat, target_names=cat_names, digits=3, zero_division=0))


In [ ]:
# --------------------------------------------------------
# 3.5 Error-pattern analysis for text classification
from sklearn.metrics import accuracy_score as acc_fn

# The winning LogReg model is near-perfect on this synthetic-but-separable
# test set (see 3.4). To still expose patterns, we (a) study the errors of the
# *weaker* baselines, and (b) look at low-confidence corners of the winner.

# (a) errors of the weaker baselines -> where does text classification fail?
mnb = MultinomialNB(alpha=0.1).fit(Xtr_t, y_train_txt)
rf = RandomForestClassifier(n_estimators=250, n_jobs=-1, random_state=42).fit(Xtr_t, y_train_txt)

weak = pd.DataFrame({
    'y': y_test_txt,
    'mnb': mnb.predict(Xte_t),
    'rf': rf.predict(Xte_t),
    'lr': yhat,
})
weak['mnb_ok'] = weak['y'] == weak['mnb']
weak['rf_ok'] = weak['y'] == weak['rf']
weak['lr_ok'] = weak['y'] == weak['lr']
print('Test accuracy  NB=%.3f  RF=%.3f  LR=%.3f' % (
    weak['mnb_ok'].mean(), weak['rf_ok'].mean(), weak['lr_ok'].mean()))

def error_pairs(df, pred_col):
    m = df[df['y'] != df[pred_col]]
    return pd.DataFrame({'true': df.loc[m.index, 'y'].map(lambda c: cat_names[c]),
                         'pred': m[pred_col].map(lambda c: cat_names[c])})

pairs_mnb = error_pairs(weak, 'mnb')
pairs_rf = error_pairs(weak, 'rf')
for nm, pr in [('MultinomialNB', pairs_mnb), ('RandomForest', pairs_rf)]:
    print(f'\n{nm} - most common true->prediction error pairs:')
    print(pr.value_counts().head(8).to_string() if len(pr) else '  (no errors on the test fold)')

# examples of NB errors (most instructive: the "hard" descriptions)
print('\nExample descriptions the weaker baseline (NB) got wrong:')
if len(pairs_mnb):
    mis_mnb = weak.index[~weak['mnb_ok']].tolist()
    rng = random.Random(7)
    for i in rng.sample(mis_mnb, min(8, len(mis_mnb))):
        print(f'  [{cat_names[weak.loc[i,"y"]]:16s}->{cat_names[weak.loc[i,"mnb"]]:16s}]  '
              f'"{df_txt.iloc[test_idx[i]].description}"')

# (b) low-confidence corners of the WINNING model (still correct, but "hard")
probs = best_txt.predict_proba(Xte_t)
conf = probs.max(axis=1)
df_te = df_txt.iloc[test_idx].copy()
df_te['confidence'] = conf
df_te['correct'] = weak['lr_ok'].values

print('\nDistribution of winner model confidence on the test fold:')
print(df_te['confidence'].describe().round(3).to_string())
print('\n10 lowest-confidence test descriptions (challenging phrasing, '
      'even when classified correctly):')
for _, r in df_te.nsmallest(10, 'confidence').iterrows():
    print(f'  conf={r["confidence"]:.3f}  true={r["category"]:16s}  "{r["description"]}"')

# (c) does description length affect confidence?
bins = pd.cut(df_te['n_words'], [0, 2, 4, 6, 8, 20])
g = df_te.groupby(bins, observed=True)['confidence'].agg(['count', 'mean'])
fig, ax = plt.subplots(figsize=(7, 4))
g['mean'].plot.bar(ax=ax, color='darkorange')
ax.set_title('Average classifier confidence vs description length')
ax.set_ylabel('mean confidence'); ax.set_xlabel('n words')
save_fig(fig, 'text_conf_by_length.png')
print('\nMean confidence by description length bucket:')
print(g.round(3).to_string())

# store variables Part 5 reuses
pairs = pairs_lr = pd.DataFrame({
    'true': [cat_names[y_test_txt[i]] for i in np.where(y_test_txt != yhat)[0]],
    'pred': [cat_names[yhat[i]] for i in np.where(y_test_txt != yhat)[0]]})
pairs_worst = (', '.join(map(str, pairs.value_counts().index[0]))
               if len(pairs) else 'none (winner is perfect on test)')
print('\nWorst winner-model error pair:', pairs_worst, ' (uses weaker baselines for the '
      'error-pattern study, since the winner is at ~100% on this data)')


In [ ]:
# --------------------------------------------------------
# 3.6 Reusable classifier function (Part 5 integration point)
def classify_description(text: str):
    """Return (category, confidence, top3) for a free-text waste description."""
    cleaned = clean_text(text)
    if not cleaned:
        raise ValueError('Empty description after cleaning.')
    X = tfidf.transform([cleaned])
    probs = best_txt.predict_proba(X)[0]
    cls_order = best_txt.classes_           # actual class indices the LR saw
    order = np.argsort(probs)[::-1]
    top = [(cat_names[cls_order[i]], float(probs[i])) for i in order[:3]]
    return cat_names[cls_order[order[0]]], float(probs[order[0]]), top

for sample in ["a crushed aluminum cola can", "soggy cardboard takeaway box with grease",
               "empty clear glass bottle", "bunch of freshly cut grass"]:
    cat, conf, top3 = classify_description(sample)
    print(f'{sample!r} -> {cat} (conf={conf:.2f})  top3={[(c, round(p,2)) for c,p in top3]}')
